# Lab 4: Scikit-Learn Preprocessing and Modeling Pipeline

**Course:** Machine Learning
**Topic:** Building a Leakage-Safe Scikit-Learn Preprocessing and Modeling Pipeline
**Duration:** 2 Hours

---

## 5. Lab Objective

The objective of this lab is to build a complete Machine Learning workflow using Scikit-Learn pipelines.

In earlier labs, we manually explored, cleaned, and transformed data. In real-world ML projects, preprocessing should not remain scattered across many manual code cells. Instead, preprocessing should be organized into a **reusable pipeline**.

**Workflow:**

```
Raw ABT
  -> Select safe features and target
  -> Remove identifiers and leakage columns
  -> Train-test split
  -> Numerical preprocessing pipeline
  -> Categorical preprocessing pipeline
  -> ColumnTransformer
  -> Modeling pipeline
  -> Prediction and evaluation
```

The main focus is **not only model accuracy**, but:
- Correct ML workflow
- Leakage prevention
- Reusable preprocessing
- Train-test consistency

## 6. Learning Outcomes

After completing this lab, students should be able to:
1. Define a clear prediction problem using an ABT dataset.
2. Separate input features `X` and target variable `y`.
3. Identify and remove identifier columns.
4. Identify and remove possible leakage columns.
5. Split data into training and testing sets **before** fitting preprocessing.
6. Build a numerical preprocessing pipeline using imputation and scaling.
7. Build a categorical preprocessing pipeline using imputation and one-hot encoding.
8. Combine preprocessing steps using `ColumnTransformer`.
9. Attach a Machine Learning model to preprocessing using `Pipeline`.
10. Evaluate a classification model using accuracy, confusion matrix, precision, recall, and F1-score.
11. Explain why pipelines reduce preprocessing mistakes and data leakage.
12. Save a trained pipeline for future use.

## 7. Prerequisites

- Basic Pandas DataFrame operations
- Missing values
- Categorical encoding
- Feature scaling
- Train-test split
- Data leakage
- Logistic regression basics
- Classification metrics basics

> **Note:** This notebook uses `data/processed/olist_orders_abt.csv`. If you have the real Analytical Base Table from Lab 2, place it at that path. Otherwise, run `scripts/generate_sample_data.py` to create a synthetic dataset with the same schema so this notebook runs end-to-end.


## Part A: Load the ABT Dataset

### Step 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import joblib

pd.set_option("display.max_columns", None)


### Step 2: Load the Dataset

Use flexible path detection so the notebook works whether it is run from the project root or from a `notebooks/` subfolder.

In [ ]:
possible_paths = [
    Path("../data/processed/olist_orders_abt.csv"),
    Path("data/processed/olist_orders_abt.csv"),
    Path("olist_orders_abt.csv"),
]

data_path = None
for path in possible_paths:
    if path.exists():
        data_path = path
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find olist_orders_abt.csv. "
        "Run scripts/generate_sample_data.py to create a synthetic ABT, "
        "or place your Lab 2 ABT at data/processed/olist_orders_abt.csv."
    )

df = pd.read_csv(data_path)
print("Loaded data from:", data_path)


### Step 3: Inspect Data

In [3]:
df.head()

,order_id,customer_id,customer_unique_id,customer_state,seller_state,seller_customer_state_match,payment_type,payment_installments,product_category,price,freight_value,freight_to_price_ratio,product_weight_g,purchase_to_estimated_days,order_purchase_date,order_estimated_delivery_date,order_delivered_customer_date,delivery_days,delivery_delay_days,review_score,review_comment_count,has_review_comment,is_low_review,is_late_delivery
0,order_000000,cust_000000,cust_unique_000223,SP,RJ,0,credit_card,8,garden,40.48,26.89,0.664,2552.0,19,2017-05-30,2017-06-18,2017-06-27,28,3,1,1,1,1,1
1,order_000001,cust_000001,cust_unique_001934,PR,MG,0,boleto,11,NaN,126.64,3.31,0.026,397.0,42,2017-10-10,2017-11-21,2017-11-21,42,-3,2,2,1,1,0
2,order_000002,cust_000002,cust_unique_001636,PE,BA,0,credit_card,10,beauty,31.42,25.03,0.797,1912.0,12,2018-12-10,2018-12-22,2018-12-24,14,1,1,0,0,1,1
3,order_000003,cust_000003,cust_unique_001097,BA,SP,0,credit_card,5,electronics,115.44,9.86,0.085,1764.0,33,2017-03-22,2017-04-24,2017-04-21,30,-3,5,0,0,0,0
4,order_000004,cust_000004,cust_unique_001082,SP,SP,1,credit_card,5,furniture,57.31,24.45,0.427,1992.0,32,2018-07-07,2018-08-08,2018-08-09,33,14,1,3,1,1,1


In [4]:
print("Shape:", df.shape)

Shape: (5000, 24)


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 24 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   order_id                       5000 non-null   str    
 1   customer_id                    5000 non-null   str    
 2   customer_unique_id             5000 non-null   str    
 3   customer_state                 5000 non-null   str    
 4   seller_state                   5000 non-null   str    
 5   seller_customer_state_match    5000 non-null   int64  
 6   payment_type                   4900 non-null   str    
 7   payment_installments           5000 non-null   int64  
 8   product_category               4750 non-null   str    
 9   price                          5000 non-null   float64
 10  freight_value                  4900 non-null   float64
 11  freight_to_price_ratio         5000 non-null   float64
 12  product_weight_g               4800 non-null   float64
 13 

In [6]:
df.describe()

,seller_customer_state_match,payment_installments,price,freight_value,freight_to_price_ratio,product_weight_g,purchase_to_estimated_days,delivery_days,delivery_delay_days,review_score,review_comment_count,has_review_comment,is_low_review,is_late_delivery
count,5000.000000,5000.000000,5000.000000,4900.000000,5000.000000,4800.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.00000
mean,0.123800,6.060800,112.261406,16.332104,0.263301,1572.013333,24.690400,24.642200,-0.466600,3.811200,0.539000,0.361800,0.197200,0.20000
std,0.329386,3.154599,72.617955,11.431653,0.770692,1097.691149,11.445101,12.263403,4.659745,1.317992,0.885797,0.480569,0.397924,0.40004
min,0.000000,1.000000,1.270000,0.290000,0.001000,6.000000,5.000000,1.000000,-5.000000,1.000000,0.000000,0.000000,0.000000,0.00000
25%,0.000000,3.000000,58.985000,7.800000,0.070000,754.750000,15.000000,15.000000,-4.000000,3.000000,0.000000,0.000000,0.000000,0.00000
50%,0.000000,6.000000,96.585000,13.855000,0.140000,1310.500000,25.000000,25.000000,-2.000000,4.000000,0.000000,0.000000,0.000000,0.00000
75%,0.000000,9.000000,148.922500,22.040000,0.280000,2135.000000,34.000000,34.000000,0.000000,5.000000,1.000000,1.000000,0.000000,0.00000
max,1.000000,11.000000,609.580000,94.110000,42.118000,7152.000000,44.000000,58.000000,14.000000,5.000000,7.000000,1.000000,1.000000,1.00000


**Student Checkpoint 1**

Answer briefly:
1. How many rows are there?
2. How many columns are there?
3. What does one row represent?
4. Which column can be used as a classification target?




## Part B: Define the Prediction Problem

### 12. Business Problem

We want to predict whether an order will be delivered late.

**Business question:** Can we predict whether an order is likely to be late using information available *before* delivery?

### 13. Target Variable

Target: `is_late_delivery`

| Value | Meaning |
|---|---|
| 0 | Order is not late |
| 1 | Order is late |

### Step 4: Check Target Column

In [7]:
target = "is_late_delivery"

if target not in df.columns:
    raise ValueError(f"Target column {target} not found in the dataset.")

df[target].value_counts()


is_late_delivery
0    4000
1    1000
Name: count, dtype: int64

In [8]:
df[target].value_counts(normalize=True) * 100

is_late_delivery
0    80.0
1    20.0
Name: proportion, dtype: float64

**Student Checkpoint 2**

Answer briefly:
1. Is the target balanced or imbalanced?
2. Why can accuracy be misleading if the target is imbalanced?
3. Which metric should we also check: precision, recall, or F1-score?

> _Your answers here..._


## Part C: Remove Identifier and Leakage Columns

### 14. Identifier Columns

Identifier columns are useful for tracking but not usually useful as predictive features.

Examples: `order_id`, `customer_id`, `customer_unique_id`

These columns may cause memorization instead of learning.

### 15. Leakage Columns

Leakage columns contain information that would not be available at prediction time, or that directly reveals the target.

If we are predicting late delivery **before** actual delivery happens, the following columns may be unsafe:

- `delivery_days` — known only after delivery
- `delivery_delay_days` — may directly define `is_late_delivery`
- `review_score` — known after delivery and customer review
- `review_comment_count`, `has_review_comment`, `is_low_review` — post-delivery information
- `order_delivered_customer_date`, `order_estimated_delivery_date` — delivery-time information

### Step 5: Define Columns to Remove

In [9]:
identifier_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id",
]

leakage_columns = [
    "delivery_days",
    "delivery_delay_days",
    "review_score",
    "review_comment_count",
    "has_review_comment",
    "is_low_review",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

columns_to_remove = identifier_columns + leakage_columns + [target]
columns_to_remove = [col for col in columns_to_remove if col in df.columns]
columns_to_remove


['order_id',
 'customer_id',
 'customer_unique_id',
 'delivery_days',
 'delivery_delay_days',
 'review_score',
 'review_comment_count',
 'has_review_comment',
 'is_low_review',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'is_late_delivery']

### Step 6: Create Feature Matrix and Target Vector

In [10]:
X = df.drop(columns=columns_to_remove)
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (5000, 12)
y shape: (5000,)


**Student Checkpoint 3**

Answer briefly:
1. Why should `order_id` not be used as a model feature?
2. Why is `delivery_delay_days` unsafe for predicting `is_late_delivery`?
3. Why is `review_score` unsafe if prediction is made before delivery?

> _Your answers here..._


## Part D: Identify Numerical and Categorical Features

### 16. Numerical and Categorical Columns

Scikit-Learn pipelines usually process numerical and categorical columns differently.

- **Numerical columns** may need: missing-value imputation, scaling
- **Categorical columns** may need: missing-value imputation, one-hot encoding

### Step 7: Identify Column Types Automatically

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


### Step 8: Check Missing Values in Selected Features

In [12]:
X[numeric_features].isnull().sum().sort_values(ascending=False).head(20)

product_weight_g               200
freight_value                  100
seller_customer_state_match      0
price                            0
payment_installments             0
freight_to_price_ratio           0
purchase_to_estimated_days       0
dtype: int64

In [13]:
X[categorical_features].isnull().sum().sort_values(ascending=False).head(20)

product_category       250
payment_type           100
customer_state           0
seller_state             0
order_purchase_date      0
dtype: int64

**Student Checkpoint 4**

Answer briefly:
1. Name three numerical features.
2. Name three categorical features.
3. Which type of feature needs one-hot encoding?
4. Which type of feature may need scaling?




## Part E: Train-Test Split

### 17. Why Split Before Preprocessing?

**Very important rule:** Split first. Fit preprocessing only on training data. Transform test data using training-fitted preprocessing.

**Wrong approach:** Fit scaler on full data -> split data
**Correct approach:** Split data -> fit scaler on training data only -> transform test data

### Step 9: Split the Data

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)


X_train shape: (4000, 12)
X_test shape: (1000, 12)
y_train shape: (4000,)
y_test shape: (1000,)


**Student Checkpoint 5**

Answer briefly:
1. Why did we use `stratify=y`?
2. Why should preprocessing not be fitted before train-test split?



## Part F: Build Numerical Preprocessing Pipeline

### 18. Numerical Pipeline

Numerical features may have missing values and different scales.

For numerical features, we will use: **Median imputation -> Standard scaling**

Median is used because many real-world business features are skewed.

### Step 10: Create Numerical Pipeline

In [15]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

numeric_pipeline


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a f

### 19. What Happens Inside the Numerical Pipeline?

**Step 1: Imputer** — The imputer fills missing numerical values.
missing numerical value -> median of training column

**Step 2: Scaler** — StandardScaler converts values into z-scores: `z = (x - mean) / std`, where mean and std are learned only from training data.

> **Important:** The median, mean, and standard deviation are learned only from training data.


## Part G: Build Categorical Preprocessing Pipeline

### 20. Categorical Pipeline

Categorical features may contain missing labels and text categories.

For categorical features, we will use: **Unknown imputation -> One-hot encoding**

### Step 11: Create Categorical Pipeline

In [16]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

categorical_pipeline


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('encoder', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",'Unknown'
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation

### 21. What Happens Inside the Categorical Pipeline?

**Step 1: Imputer** — Missing category values are replaced with `"Unknown"`.

**Step 2: One-Hot Encoder** — Text categories are converted into binary columns.

| payment_type | credit_card | boleto | voucher |
|---|---|---|---|
| credit_card | 1 | 0 | 0 |
| boleto | 0 | 1 | 0 |
| voucher | 0 | 0 | 1 |

**Why `handle_unknown="ignore"`?** In real-world data, new categories may appear in test or production data. `handle_unknown="ignore"` prevents the model from failing.


## Part H: Combine Pipelines Using ColumnTransformer

### 22. What is ColumnTransformer?

`ColumnTransformer` applies different preprocessing steps to different columns:
- Numerical columns -> numerical pipeline
- Categorical columns -> categorical pipeline

This is useful because different data types need different treatment.

### Step 12: Create Preprocessor

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

**Student Checkpoint 6**

Answer briefly:
1. Why do numerical and categorical columns need different preprocessing?
2. What is the role of `ColumnTransformer`?
3. Why is `handle_unknown="ignore"` useful?

> _Your answers here..._


## Part I: Create Complete Modeling Pipeline

### 23. Add Model to the Pipeline

We will use **Logistic Regression** as the first baseline classifier.

**Why Logistic Regression?**
- Simple
- Fast
- Interpretable
- Good baseline for classification
- Useful for understanding complete ML workflow

### Step 13: Create Full Pipeline

In [18]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

model_pipeline


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

### 24. Why Use `class_weight="balanced"`?

If late deliveries are less common than non-late deliveries, the model may prefer the majority class. `class_weight="balanced"` gives more weight to the minority class. This can improve recall for rare classes.


## Part J: Train the Pipeline

### Step 14: Fit the Pipeline

This single line performs:
- Fit numerical imputer on training data
- Fit numerical scaler on training data
- Fit categorical imputer on training data
- Fit one-hot encoder on training data
- Transform training data
- Train LogisticRegression model

> **Important:** All fitting happens only on training data.


In [19]:
model_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## Part K: Make Predictions

### Step 15: Predict on Test Data

This single line performs:
- Apply training-fitted preprocessing to test data
- Make predictions using trained model


In [20]:
y_pred = model_pipeline.predict(X_test)

### Step 16: Predicted Probabilities

Predicted probability means: estimated probability that the order will be late.

In [21]:
y_pred_proba = model_pipeline.predict_proba(X_test)[:, 1]
y_pred_proba[:10]


array([0.88298145, 0.14522538, 0.27063766, 0.82175275, 0.07600807,
       0.13583451, 0.05545695, 0.11525106, 0.7376814 , 0.5607385 ])

## Part L: Evaluate the Model

### 25. Accuracy

Accuracy means: correct predictions / total predictions

But accuracy alone may be misleading for imbalanced data.


In [22]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


Accuracy: 0.759


### 26. Confusion Matrix

Interpretation for binary classification:

|  | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | True Negative | False Positive |
| **Actual 1** | False Negative | True Positive |


In [23]:
cm = confusion_matrix(y_test, y_pred)
print(cm)


[[632 168]
 [ 73 127]]


### 27. Classification Report

This report includes: precision, recall, F1-score, support

In [24]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.90      0.79      0.84       800
           1       0.43      0.64      0.51       200

    accuracy                           0.76      1000
   macro avg       0.66      0.71      0.68      1000
weighted avg       0.80      0.76      0.77      1000



### 28. Understanding Precision, Recall, and F1-Score

**Precision** answers: Out of all orders predicted late, how many were actually late?

**Recall** answers: Out of all actually late orders, how many did the model catch?

**F1-Score** balances precision and recall. Use F1-score when the target is imbalanced.

**Student Checkpoint 7**

Answer briefly:
1. What does accuracy measure?
2. Why can accuracy be misleading?
3. What does recall measure?
4. Why may recall be important for late delivery prediction?

> _Your answers here..._


## Part M: Inspect Transformed Feature Names

### 29. Get Feature Names After Preprocessing

After one-hot encoding, the number of columns increases.


In [25]:
feature_names = model_pipeline.named_steps["preprocessor"].get_feature_names_out()
feature_names[:20]


array(['num__seller_customer_state_match', 'num__payment_installments',
       'num__price', 'num__freight_value', 'num__freight_to_price_ratio',
       'num__product_weight_g', 'num__purchase_to_estimated_days',
       'cat__customer_state_BA', 'cat__customer_state_CE',
       'cat__customer_state_GO', 'cat__customer_state_MG',
       'cat__customer_state_PE', 'cat__customer_state_PR',
       'cat__customer_state_RJ', 'cat__customer_state_RS',
       'cat__customer_state_SC', 'cat__customer_state_SP',
       'cat__seller_state_BA', 'cat__seller_state_MG',
       'cat__seller_state_PR'], dtype=object)

In [26]:
len(feature_names)

766

### 30. Why Did the Number of Features Increase?

Categorical variables are expanded into multiple binary columns. For example, `customer_state` may become `customer_state_MP`, `customer_state_MH`, `customer_state_KA`, ...


## Part N: Save the Complete Pipeline

### 31. Why Save the Pipeline?

In real ML systems, we need to save **preprocessing steps + model**, not only the model. If we save only the model, future raw data will not be transformed correctly.

### Step 17: Save the Pipeline

If your notebook is in the project root folder, use the second (non-relative) path instead.


In [27]:
output_model_path = Path("../models/late_delivery_pipeline.joblib")
output_model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(model_pipeline, output_model_path)
print("Pipeline saved at:", output_model_path)


Pipeline saved at: ../models/late_delivery_pipeline.joblib


### Step 18: Load the Pipeline Again

In [28]:
loaded_pipeline = joblib.load(output_model_path)
loaded_predictions = loaded_pipeline.predict(X_test)
loaded_predictions[:10]


array([1, 0, 0, 1, 0, 0, 0, 0, 1, 1])

## Part O: Predict on New Data

### 32. Create a Sample New Order

Use one row from the test set as a sample new order.


In [29]:
sample_order = X_test.iloc[[0]]
sample_order


,customer_state,seller_state,seller_customer_state_match,payment_type,payment_installments,product_category,price,freight_value,freight_to_price_ratio,product_weight_g,purchase_to_estimated_days,order_purchase_date
4319,CE,PR,0,credit_card,3,toys,19.69,31.66,1.608,1972.0,44,2017-05-13


In [30]:
loaded_pipeline.predict(sample_order)

array([1])

In [31]:
loaded_pipeline.predict_proba(sample_order)[:, 1]

array([0.88298145])

**Interpretation:**
- Probability close to 1 -> likely late
- Probability close to 0 -> likely not late


## Part P: Common Errors and Fixes

### 33. Error: Target Column Not Found

**Possible reason:** Lab 2 ABT did not create `is_late_delivery`.

**Fix:** Inspect `df.columns`, then create the target if required using Lab 2 logic.

### 34. Error: Could Not Find CSV File

**Possible reason:** Wrong relative path.

**Fix:**
```python
import os
os.getcwd()
```
Then adjust the path.

### 35. Error: Unknown Category During Prediction

**Fix:** `OneHotEncoder(handle_unknown="ignore")`

### 36. Error: Logistic Regression Did Not Converge

**Fix:** `LogisticRegression(max_iter=2000)` or scale numerical features properly.

### 37. Error: Too Many Features After One-Hot Encoding

**Reason:** High-cardinality categorical columns.

**Fix:**
- Remove ID-like columns
- Group rare categories
- Limit top categories
- Use hashing or target encoding later


## Part Q: Complete Code in One Place

Use this section after understanding all individual parts above. It reproduces the full workflow end-to-end in a single block.


In [ ]:
# 1. Load data
possible_paths = [
    Path("../data/processed/olist_orders_abt.csv"),
    Path("data/processed/olist_orders_abt.csv"),
    Path("olist_orders_abt.csv"),
]
data_path = None
for path in possible_paths:
    if path.exists():
        data_path = path
        break
if data_path is None:
    raise FileNotFoundError("Could not find olist_orders_abt.csv. Please check the file path.")

df = pd.read_csv(data_path)

# 2. Define target
target = "is_late_delivery"
if target not in df.columns:
    raise ValueError(f"Target column {target} not found in the dataset.")

# 3. Remove identifiers and leakage columns
identifier_columns = ["order_id", "customer_id", "customer_unique_id"]
leakage_columns = [
    "delivery_days",
    "delivery_delay_days",
    "review_score",
    "review_comment_count",
    "has_review_comment",
    "is_low_review",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
columns_to_remove = identifier_columns + leakage_columns + [target]
columns_to_remove = [col for col in columns_to_remove if col in df.columns]

X = df.drop(columns=columns_to_remove)
y = df[target]

# 4. Identify column types
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# 5. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 6. Numerical pipeline
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# 7. Categorical pipeline
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

# 8. ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ]
)

# 9. Full model pipeline
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

# 10. Train
model_pipeline.fit(X_train, y_train)

# 11. Predict
y_pred = model_pipeline.predict(X_test)

# 12. Evaluate
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 13. Save pipeline
output_model_path = Path("../models/late_delivery_pipeline.joblib")
output_model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model_pipeline, output_model_path)
print("\nPipeline saved at:", output_model_path)


## Part R: Lab Deliverables

Students must submit:
1. Completed Jupyter Notebook.
2. Screenshot/output of dataset shape and columns.
3. Target distribution table.
4. List of removed identifier columns.
5. List of removed leakage columns.
6. List of numerical and categorical features.
7. Code for numerical pipeline.
8. Code for categorical pipeline.
9. Code for `ColumnTransformer`.
10. Code for complete model pipeline.
11. Accuracy, confusion matrix, and classification report.
12. Short explanation of why pipeline prevents leakage.
13. Saved model pipeline file: `late_delivery_pipeline.joblib`


## Part S: Student Reflection Questions

Answer the following:
1. What is the main purpose of a Scikit-Learn pipeline?
2. Why should preprocessing be fitted only on training data?
3. What is the role of `ColumnTransformer`?
4. Why do numerical and categorical columns need different preprocessing?
5. Why should identifiers not be used as features?
6. Why is `delivery_delay_days` a leakage column?
7. Why is `OneHotEncoder(handle_unknown="ignore")` useful?
8. Why did we use median imputation for numerical columns?
9. Which algorithms are sensitive to scaling?
10. Why should we save the complete pipeline and not only the model?



## Part T: Assessment Rubric

| Component | Marks |
|---|---|
| Dataset loading and inspection | 1 |
| Correct target selection | 1 |
| Removal of identifier and leakage columns | 1 |
| Correct train-test split | 1 |
| Numerical preprocessing pipeline | 1 |
| Categorical preprocessing pipeline | 1 |
| Correct use of `ColumnTransformer` | 1 |
| Complete model pipeline | 1 |
| Model evaluation | 1 |
| Reflection and interpretation | 1 |
| **Total** | **10** |


## Part U: Extension Task

If time remains, try the following.

### 1. Replace Logistic Regression with Random Forest

In [33]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",
    )),
])

rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
print(classification_report(y_test, rf_pred))


              precision    recall  f1-score   support

           0       0.83      0.98      0.90       800
           1       0.68      0.19      0.30       200

    accuracy                           0.82      1000
   macro avg       0.75      0.58      0.60      1000
weighted avg       0.80      0.82      0.78      1000



### 2. Compare Logistic Regression and Random Forest

| Model | Accuracy | Precision | Recall | F1-score |
|---|---|---|---|---|
| Logistic Regression | | | | |
| Random Forest | | | | |


In [34]:
from sklearn.metrics import precision_score, recall_score, f1_score

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, rf_pred),
    ],
    "Precision": [
        precision_score(y_test, y_pred),
        precision_score(y_test, rf_pred),
    ],
    "Recall": [
        recall_score(y_test, y_pred),
        recall_score(y_test, rf_pred),
    ],
    "F1-score": [
        f1_score(y_test, y_pred),
        f1_score(y_test, rf_pred),
    ],
})
comparison


,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,0.759,0.430508,0.635,0.513131
1,Random Forest,0.820,0.678571,0.190,0.296875


### 3. Discussion

Answer:
1. Which model performed better?
2. Which model is easier to interpret?
3. Does Random Forest require scaling?
4. Why did we still keep preprocessing in the pipeline?



## Part V: Key Takeaways

In this lab, we learned that:

1. A Machine Learning project needs both preprocessing and modeling.
2. Manual preprocessing can cause leakage and inconsistency.
3. Pipelines organize preprocessing and model training.
4. `ColumnTransformer` applies different preprocessing to different column types.
5. Numerical features often need imputation and scaling.
6. Categorical features need imputation and encoding.
7. Leakage columns must be removed before training.
8. The complete pipeline should be saved for future prediction.
9. A good ML workflow is reproducible.
10. A reliable model starts with a reliable pipeline.

### Final Message

A trained model alone is not a complete Machine Learning system.

A complete ML system is:

**Preprocessing + Model + Evaluation + Reproducibility**

In professional Machine Learning, pipelines are not optional. They are essential.

---
*Prepared by: Sharad Laad | ORY AI Labs*
